<a href="https://colab.research.google.com/github/yenlung/Generative-AI/blob/main/RAG_01_%E6%89%93%E9%80%A0%E4%BD%A0%E7%9A%84_RAG_%E7%B3%BB%E7%B5%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG01｜戰錘 40K 第11版：官方 PDF → 向量知識庫
作者：賴品達（請自行核對學籍資料）。改編自蔡炎龍老師 RAG01。
**AI 協作修改**：PDF 網址下載、依頁分段、來源稽核與輸出 rag-data.zip。
執行順序：逐格執行，核對抽出的文字，再建立向量；本份不使用 Groq。
不自動宣稱資料是最新版本。多欄、表格、圖片地形仍需對照原 PDF。

## 1. 安裝套件

我們使用 Sentence Transformers 載入模型、NumPy 處理向量，以及 FAISS 搜尋向量。這格固定本教材採用的套件版本，方便記錄與重現。

先執行本格，再執行下格的 import。如果你已在本次執行階段載入其他版本的相關套件，請在安裝後重新啟動工作階段，再從下一格繼續；不需要重跑安裝格。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"
%pip install -q pymupdf requests


## 2. 檢查環境與集中設定

`BATCH_SIZE` 可以調小以降低記憶體使用量。`EXPECTED_DOCUMENT_COUNT` 只用來提醒教材篇數；換自己的資料時可設為 `None`，不會強制只能有16篇。

以下會列出套件版本和 CPU／GPU。Colab 常見的 T4 不支援原生 bfloat16，本程式會改用 float32；不使用 EmbeddingGemma 2 不相容的 float16。

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
EXPECTED_DOCUMENT_COUNT = None  # 換自己的資料的大小
EXPORT_ZIP_NAME = "rag-data.zip"

# text 本身已含標題；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

## 重點 A：從指定網址下載六份官方 PDF
每個下載結果都列出。預設有任何來源失敗即停止，先修正網址；若你明確要用部分文件，可改 ALLOW_PARTIAL=True。Wahapedia 暫作外部核對連結，不把首頁或舊版資料匯入。

In [ ]:
SOURCES = [{'source_id': 'core_en', 'title': '核心規則（英文）', 'language': 'en', 'kind': 'core', 'url': 'https://assets.warhammer-community.com/eng_01-06_warhammer40k_new40k_core_rules-was6fbu1ix-hfewhmxyiy.pdf', 'enabled': True}, {'source_id': 'updates_en', 'title': '通用規則更新（英文）', 'language': 'en', 'kind': 'update', 'url': 'https://assets.warhammer-community.com/eng_wh40k_core&key_universal_rules_updates-lu3grocned-rphh78bl6k.pdf', 'enabled': True}, {'source_id': 'core_zh', 'title': '核心規則（簡體中文）', 'language': 'zh', 'kind': 'core', 'url': 'https://assets.warhammer-community.com/chi_01-06_warhammer40k_new40k_core_rules-gihrxgzhgo-iickazpeog.pdf', 'enabled': True}, {'source_id': 'updates_zh', 'title': '通用規則更新（簡體中文）', 'language': 'zh', 'kind': 'update', 'url': 'https://assets.warhammer-community.com/chi_wh40k_core&key_universal_rules_updates-jldybkdif4-wfbo8jewfg.pdf', 'enabled': True}, {'source_id': 'event_en', 'title': '賽事指南（英文）', 'language': 'en', 'kind': 'event', 'url': 'https://assets.warhammer-community.com/eng_14-10_wh40k_event_companion-rdwnqprl4c-nj8oqogadv.pdf', 'enabled': True}, {'source_id': 'event_zh', 'title': '賽事指南（簡體中文）', 'language': 'zh', 'kind': 'event', 'url': 'https://assets.warhammer-community.com/chi_14-10_wh40k_event_companion-0lkqm4w0bw-pwbbufksvw.pdf', 'enabled': True}]
ALLOW_PARTIAL = False
import requests, fitz
from google.colab import files
DATA_DIR = Path(tempfile.mkdtemp(prefix="wh40k_pdf_"))
audit=[]; downloaded=[]
for src in SOURCES:
    if not src['enabled']: continue
    try:
        response=requests.get(src['url'],timeout=(15,120))
        response.raise_for_status()
        if not response.content.startswith(b'%PDF'): raise ValueError('回傳內容不是 PDF')
        path=DATA_DIR/(src['source_id']+'.pdf');path.write_bytes(response.content)
        with fitz.open(path) as pdf: pages=len(pdf)
        digest=hashlib.sha256(response.content).hexdigest()
        downloaded.append((src,path,digest))
        audit.append({**src,'status':'成功','pages':pages,'sha256':digest})
    except Exception as exc:
        audit.append({**src,'status':'失敗','error':str(exc)[:200]})
display(pd.DataFrame(audit))
if not downloaded: raise RuntimeError('全部來源下載失敗，請確認網址或網路。')
if any(x['status']=='失敗' for x in audit) and not ALLOW_PARTIAL:
    raise RuntimeError('部分 PDF 未下載成功。修正網址重跑，或明確允許部分資料後再建庫。')
SOURCE_ARCHIVE_SHA256=hashlib.sha256(json.dumps(audit,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
archive_name='user_selected_official_pdf_urls'


## 重點 B：文字擷取與分段
保留 PDF 實體頁碼（從1開始，未必等於書上印刷頁碼）、來源、語言與文件類型。以區塊座標排序，仍需人工核對多欄閱讀順序。每段600字元、重疊100字元是初始設定，可依測試修改。低文字量頁面會列出，不能據此回答圖片尺寸。

In [ ]:
REVIEWED_ZH_SHA256 = '18e40276dc81034357d6b7a852b9ce808e0094bc6263e849ef8849b8d210198b'
CHUNK_CHARS=600
OVERLAP_CHARS=100
def split_text(value,size=600,overlap=100):
    if not 0 <= overlap < size: raise ValueError('需滿足 0 <= overlap < size')
    for start in range(0,len(value),size-overlap):
        piece=value[start:start+size].strip()
        if piece: yield piece
        if start+size>=len(value): break
documents=[]; weak_pages=[]
for src,path,digest in downloaded:
    with fitz.open(path) as pdf:
        for pno,page in enumerate(pdf,start=1):
            body=page.get_text('text',sort=True).strip()
            if len(body)<30:
                weak_pages.append({'source':src['source_id'],'pdf_page':pno,'characters':len(body)})
                continue
            # AI協作：此補充已在本次PDF第18頁圖像逐項核對；僅對相同SHA256來源生效。
            if src['source_id']=='core_zh' and pno==18 and digest==REVIEWED_ZH_SHA256:
                body += '\n【第18頁骰子圖示之人工核對補充】命中擲骰：未修正的1失敗；未修正的6暴擊命中；其他按該次攻擊的BS/WS判斷。致傷擲骰：未修正的1失敗；未修正的6暴擊致傷。力量至少是韌性兩倍：2+；力量大於韌性：3+；力量等於韌性：4+；力量小於韌性：5+；力量不高於韌性一半：6+。按原表由上而下第一個符合條件判斷。此補充僅核對該頁，不代表其他圖表已核對。'
            for chunk_no,piece in enumerate(split_text(body,CHUNK_CHARS,OVERLAP_CHARS),start=1):
                title=f"{src['title']}｜PDF第{pno}頁｜段{chunk_no}"
                metadata={'title':title,'source_url':src['url']+f'#page={pno}',
                    'date':None,'retrieved_at':datetime.now(ZoneInfo('Asia/Taipei')).isoformat(),
                    'filename':path.name,'document_type':'官方PDF文字擷取（非人工校訂）',
                    'edition':'11th（依使用者指定資料集）','language':src['language'],
                    'kind':src['kind'],'pdf_page':pno,'chunk_no':chunk_no,
                    'source_id':src['source_id'],'sha256':digest}
                documents.append({'doc_id':f'doc_{len(documents)+1:05d}',
                                  'text':title+'\n'+piece,'metadata':metadata})
if not documents: raise ValueError('沒有可用正文；掃描PDF需先OCR。')
index_doc_ids=[d['doc_id'] for d in documents]
print('段落數：',len(documents),'／低文字量頁面數：',len(weak_pages))
display(pd.DataFrame(weak_pages))
display(pd.DataFrame([{'doc_id':d['doc_id'],**d['metadata']} for d in documents]))
print('請對照原PDF核對第一段：\n',documents[0]['text'])


本範例改由 PDF 擷取流程建立正文與 metadata，來源日期未標明時保留空值。

In [ ]:
# 本步已整合至上方 PDF 擷取流程。


本範例改由 PDF 擷取流程建立正文與 metadata，來源日期未標明時保留空值。

In [ ]:
# 本步已整合至上方 PDF 擷取流程。


本範例改由 PDF 擷取流程建立正文與 metadata，來源日期未標明時保留空值。

In [ ]:
# 本步已整合至上方 PDF 擷取流程。


## 8. 載入 EmbeddingGemma 2

先查詢模型目前的確切版本（commit），再載入該版本，將版本記錄到輸出設定，方便第二份 Colab 使用相同模型。只啟用文字部分，使用適合裝置的精度。

首次下載需要等待。本步使用公開模型，不讀取 Groq 或其他密鑰。若下載失敗，先確認網路；模型架構不受支援時，確認安裝格已完成，並重新啟動工作階段。程式不會默默改用別的模型。

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

## 9. 檢查輸入長度，避免靜默截斷

本模型採用不同的任務前綴來區分文件和問題。文件本身已有標題，所以組成 `title: none | text: {text}`；問題則用 `task: search result | query: {question}`。

下格計算實際送入模型的文件文字（含前綴）長度。超過上限便停止，不會只取前半段。換成長文件時，可以在後續課程加入 chunking，或重新設計文件單位。

In [ ]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts})
display(length_table)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("本教材使用的 token 上限：", TOKEN_LIMIT)
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])





## 10. 產生並觀察文件向量

一篇文件對應矩陣中的一列，欄數是向量維度。`prompt=""` 表示前綴已自行加入，不讓模型額外再加一次。

向量正規化後長度約為1，內積便等於 cosine similarity。下格也將資料轉為 FAISS 使用的 float32 連續陣列，檢查空向量、非有限值及形狀。

In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))

## 11. 建立 FAISS 精確搜尋索引

`IndexFlatIP` 比較問題向量與所有文件向量的內積。因為雙方都正規化，排名就是 cosine similarity 的排名。

FAISS 回傳的是向量編號，**不是原文**。`index_doc_ids` 保存向量順序與文件編號的對應，再透過 `doc_id` 找回正文與 metadata。

In [ ]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

## 12. 定義問題向量化與搜尋函式

問題也要用同一個 embedding 模型向量化，但使用查詢專用前綴。`top_k` 表示取回幾篇文件；程式限制它不超過實際文件數。

搜尋函式只做檢索，還沒有呼叫 LLM。精確搜尋保證依目前向量找到分數最高者，不保證文件一定足以回答問題。

In [ ]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 13. 試查一次，閱讀真正取回的內容

修改 `QUESTION` 和 `TOP_K`，重新執行本格即可查詢。先看標題及分數，再讀完整正文，判斷是否有足夠依據。

可以比較 `top_k=1`、`3`、`5`。也可問「政大附近哪家飲料店時薪最高？」：即使教材沒有答案，FAISS 仍會傳回最相近的文章。**相似度不是正確率，也不是模型有答案的機率。**

In [ ]:
QUESTION = "請解釋命中、致傷與豁免的基本攻擊流程。"
TOP_K = 5
results = search_documents(QUESTION, TOP_K)
display(pd.DataFrame([
    {"排名": r["rank"], "doc_id": r["doc_id"], "標題": r["metadata"]["title"],
     "cosine_similarity": r["score"], "來源": r["metadata"]["source_url"]}
    for r in results
]))
for result in results:
    print(f"\n[{result['doc_id']}] {result['metadata']['title']} ({result['score']:.4f})")
    print(result["text"])
    print("來源：", result["metadata"]["source_url"] or "未提供")

## 14. 保存索引、文件、編號對照與設定

第二份 Colab 需要下面四個檔案，不能只拿 FAISS 索引：

| 檔案 | 用途 |
|---|---|
| `index.faiss` | 文件向量與搜尋索引 |
| `documents.json` | 取回後提供給 LLM 的正文與來源資訊 |
| `index_doc_ids.json` | FAISS 編號對應的文件編號 |
| `config.json` | 模型確切版本、前綴、維度、正規化、套件版本及檢查資訊 |

不保存模型權重、API 金鑰或其他環境變數。第二份 Colab 載入相同模型版本，只重新計算問題向量。

In [ ]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "pdf_page_character_chunks",
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")
print("\n模型版本及前綴已寫入 config.json。")
config["source_audit"]=audit
config["extraction_warnings"]=weak_pages
(EXPORT_DIR / "config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")


## 15. 重新載入，驗證第二份 Colab 所需的資料

從剛才保存的檔案重新載入，檢查完整性、文件編號與向量對應，再使用**同一個問題向量**比對保存前後的排名與分數。此處不重算問題向量，以免不同運算裝置的細微浮點誤差干擾索引保存檢查。

也將 FAISS 結果與 NumPy 的全矩陣內積排序比較，確認精確搜尋的意義。遇到完全同分時，兩套工具的同分排列順序可以不同。

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

## 16. 打包並下載知識庫

ZIP 根目錄包含前述四個檔案。下載後，留給第二份 Colab 上傳使用；模型權重不在 ZIP 裡。

如果你更換文件、修改擷取函式或 embedding 設定，請重新執行受影響步驟並重新匯出，不要混用舊索引與新文件。

In [ ]:
EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
files.download(str(EXPORT_ZIP))

## 完成後
下載 rag-data.zip。先核對搜尋結果與原 PDF，再將 ZIP 放到你自己的公開下載位置。把直連網址填入 RAG02 的 RAG_DATA_URL。此時還沒有產生任何語言模型回答。固定套件匯入保留自老師範例。
來源：https://github.com/yenlung/Generative-AI/blob/main/RAG_01_打造你的_RAG_系統.ipynb